## 01 - FHIR API -> raw

Pulls one resource type from the FHIR server and stores every response page as it came back:

`Files/raw/fhir/<Resource>/<yyyy>/<mm>/<dd>/<run_id>/<Resource>_<yyyymmdd>_p0001.json`

- incremental on `_lastUpdated`: starts at the `api` watermark in `ctl_watermark` (first run goes back `initial_lookback_days`), one window per UTC day
- paging just follows the Bundle `next` link until there is none
- every HTTP call is written to `ctl_api_call_log` (url + params, when it was called, when the file was saved, status, retries)

The pipeline runs this once per resource: Patient -> Encounter -> Observation -> Condition.

In [ ]:
resource_type = "Patient"
run_id = ""            # pipeline passes @pipeline().RunId
start_date = ""        # optional reload start (UTC), e.g. "2026-11-09"
end_date = ""          # optional, default is now minus the safety lag
response_format = ""   # json or xml, blank: from config
base_url = ""          # blank: from config
max_pages = -1         # -1: from config, 0: no limit
config_path = "/lakehouse/default/Files/config/pipeline_config.json"
code_path = "/lakehouse/default/Files/code"
lakehouse_root = ""    # blank: use the default lakehouse attached to the notebook

In [ ]:
import json
import sys

if code_path and code_path not in sys.path:
    sys.path.insert(0, code_path)

from pyspark.sql import functions as F

from fhir_lakehouse.config import apply_lakehouse_root, ctl_table, get_resource, load_config
from fhir_lakehouse.control import StepLog, new_run_id
from fhir_lakehouse.ingest import ingest_resource
from fhir_lakehouse.storage import Lakehouse, get_spark

overrides = {"fhir.format": response_format, "fhir.base_url": base_url}
if int(max_pages) >= 0:
    overrides["fhir.max_pages_per_window"] = int(max_pages)
cfg = apply_lakehouse_root(load_config(config_path, overrides), lakehouse_root)
resource = get_resource(cfg, resource_type)
run_id = run_id or new_run_id()

spark = get_spark()
spark.conf.set("spark.sql.session.timeZone", cfg["spark"]["timezone"])
lh = Lakehouse(spark, cfg)

print(f"run_id={run_id} resource={resource['name']} format={cfg['fhir']['format']} api={cfg['fhir']['base_url']}")

In [ ]:
step = StepLog(lh, cfg, run_id, "ingest_raw", resource["name"])
try:
    stats = ingest_resource(lh, cfg, resource, run_id, start=start_date, end=end_date)
    step.finish("SUCCEEDED", rows_read=stats["pages"], rows_written=stats["resources"], details=stats)
except Exception as e:
    step.finish("FAILED", error=repr(e))
    raise

In [ ]:
# what got called in this run
if lh.exists(ctl_table(cfg, "api_call_log")):
    (lh.read(ctl_table(cfg, "api_call_log"))
       .where((F.col("run_id") == run_id) & (F.col("resource_type") == resource["name"]))
       .select("window_start", "page_number", "entry_count", "http_status", "attempts", "called_at", "saved_at", "raw_path")
       .orderBy("window_start", "page_number")
       .show(10, truncate=False))

In [ ]:
result = json.dumps(stats, default=str)
print(result)
try:
    notebookutils.notebook.exit(result)  # shows up as the activity output in the pipeline
except NameError:
    pass  # not running in Fabric